<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 t-SNE: El Mapa de Amigos de una Fiesta 🎉
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/05%20-%20Reduccion%20de%20Dimensionalidad/Para%20Dummies/01_tSNE_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 01 (t-SNE)](../01_tSNE.ipynb). *t-SNE* es un método para **dibujar en un plano** datos que tienen muchísimas columnas, de modo que **los parecidos queden juntos**. Es famoso porque produce mapas muy llamativos, pero también porque **se interpreta mal con facilidad**. Lo vamos a entender con una historia: **sentar a los invitados de una fiesta**.

Al terminar podrás explicar:
1. **Qué intenta conservar** t-SNE (los amigos de cada quien, no las distancias grandes).
2. Qué es la **perplejidad** (cuántos amigos cuenta cada persona).
3. Las **trampas**: el tamaño de los grupos y la distancia entre ellos **no significan nada**, y el mapa **cambia con la semilla**.
4. Por qué t-SNE sirve para **explorar** pero no para «predecir» ni para clasificar datos nuevos.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../01_tSNE.ipynb). Antes: [PCA (Dummies)](00_PCA_Dummies.ipynb).

---
## 1. La fiesta con 500 invitados 🎉

Tienes 500 invitados y un salón **plano** (el papel). Cada invitado tiene un perfil descrito con **64 datos** (imagina 64 gustos y rasgos). Quieres sentarlos para que **cada uno quede cerca de sus amigos**. No te importa tanto qué tan lejos queden las mesas entre sí: lo que importa es **quién está al lado de quién**.

Eso es *t-SNE*: para cada persona mide **a quién consideraría su vecino** en el mundo de 64 datos, y luego mueve a todos en el plano hasta que **las vecindades del plano se parezcan a las de los 64 datos**. Aquí los «invitados» son 500 dibujos de números (0 al 9) de 64 píxeles. Comparamos con *PCA*, que solo **proyecta con rectas**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

dig = load_digits()
rng = np.random.RandomState(42)
idx = np.concatenate([rng.choice(np.where(dig.target == c)[0], 50, replace=False) for c in range(10)])
X, y = dig.data[idx] / 16.0, dig.target[idx]                     # 500 dibujos (50 de cada numero)

mapa_pca = PCA(n_components=2).fit_transform(X)
mapa_tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=0).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for ax, mapa, titulo in zip(axes, [mapa_pca, mapa_tsne], ["PCA (una sombra recta):\nlos números se mezclan", "t-SNE (mapa de amigos):\ncada número forma su mesa"]):
    sc = ax.scatter(mapa[:, 0], mapa[:, 1], c=y, cmap="tab10", s=14)
    ax.set_title(titulo, fontweight="bold"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, label="número que representa el dibujo", shrink=0.8); plt.show()
print("Ninguno de los dos métodos vio las etiquetas (los colores): los grupos aparecen solo porque los dibujos parecidos quedan cerca.")

---
## 2. ¿Cuántos amigos cuenta cada persona? La perplejidad 🎚️

*t-SNE* tiene un botón muy importante: la **perplejidad**. Es, más o menos, **cuántos amigos cercanos tiene en cuenta cada invitado** al decidir dónde sentarse.

- **Pocos amigos (perplejidad baja):** solo importan los 2 o 3 más cercanos; el mapa se llena de **islitas pequeñas**, a veces inventadas.
- **Muchos amigos (perplejidad alta):** cada persona mira un círculo grande; los grupos se **juntan** y se ve más el panorama general.

No hay un valor «correcto» (lo típico es entre 5 y 50). Lo sensato es **probar varios** y confiar solo en lo que **se repite**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, perp in zip(axes, [5, 30, 100]):
    m = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto", random_state=0).fit_transform(X)
    ax.scatter(m[:, 0], m[:, 1], c=y, cmap="tab10", s=12)
    ax.set_title(f"perplejidad = {perp}\n({'pocos' if perp == 5 else 'algunos' if perp == 30 else 'muchos'} amigos por persona)", fontweight="bold"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Los mismos datos, tres mapas distintos: la perplejidad cambia el mapa. Lo que se repite en los tres es lo que merece confianza.")

---
## 3. Cuidado: no es un mapa de carreteras 🗺️

Un mapa de *t-SNE* **parece** un mapa geográfico, pero **no se lee igual**. Lo que **sí** puedes creer: *«estos dibujos son parecidos entre sí»*. Lo que **no** debes creer:

| Tentación | Por qué es un error | Ejemplo cotidiano |
|---|---|---|
| «Este grupo es **más grande**, tiene más variedad» | El método **ajusta** el tamaño de cada grupo; grupos muy distintos en dispersión salen **parecidos** | Una mesa de 8 amigos muy unidos y una de 8 desconocidos se dibujan del mismo tamaño |
| «Estos grupos están **más lejos**, son más distintos» | La distancia **entre** mesas no se cuida | Dos mesas en esquinas opuestas del salón podrían ser vecinas en la vida real |
| «Siempre sale igual» | Depende de la **suerte inicial** (la *semilla*) | Si sientas a la gente al azar al empezar, cada vez terminan en otra posición |
| «Veo grupos, luego hay grupos» | En **puros números al azar** también aparecen «grupos» con perplejidad baja | Ver caras en las nubes |
| «Lo uso con un invitado nuevo» | *t-SNE* **no aprende una regla**: para un invitado nuevo hay que **rehacer todo el mapa** | No hay «un plano guardado» para sentar al que llega tarde |

Veámoslo: (1) tres grupos con **dispersión muy distinta**, (2) los mismos dibujos con **dos semillas** distintas.

In [ ]:
# (1) Tres grupos: muy compactos, medianos y muy dispersos (en 10 columnas)
Xg, yg = make_blobs(n_samples=450, n_features=10, centers=3, cluster_std=[0.5, 1.0, 4.0], center_box=(-30, 30), random_state=42)
mg = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=0).fit_transform(Xg)
radio = lambda A: np.sqrt(((A - A.mean(axis=0)) ** 2).sum(axis=1).mean())            # que tan "regado" esta un grupo
print("Dispersión REAL de los 3 grupos (compacto, mediano, muy disperso):", [round(float(radio(Xg[yg == c])), 1) for c in range(3)])
print("Tamaño de cada grupo EN EL MAPA                                :", [round(float(radio(mg[yg == c])), 1) for c in range(3)], " <- casi iguales")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].scatter(mg[:, 0], mg[:, 1], c=yg, cmap="tab10", s=12); axes[0].set_title("(1) Dispersiones reales muy distintas...\npero tamaños casi iguales en el mapa", fontweight="bold")
# (2) Mismos dibujos, dos semillas (inicio al azar) distintas
for ax, semilla in zip(axes[1:], [0, 1]):
    m = TSNE(n_components=2, perplexity=30, init="random", learning_rate="auto", random_state=semilla).fit_transform(X)
    ax.scatter(m[:, 0], m[:, 1], c=y, cmap="tab10", s=12); ax.set_title(f"(2) Mismos dibujos, semilla {semilla}:\nlas mesas cambian de lugar", fontweight="bold")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Lo que se mantiene: quién está cerca de quién. Lo que cambia: dónde queda cada mesa y qué tan lejos de las otras.")

---
##### 🛠️ Práctica 1: ¿Qué cambia cuando cambias la suerte inicial?

En la sección 3 viste que *t-SNE* empieza sentando a los invitados al azar, y que esa «suerte inicial» (la *semilla*) cambia el mapa. Vamos a comprobarlo con los 500 dibujos `X`:

1. Dibuja **tres mapas** con `TSNE(n_components=2, perplexity=30, init="random", learning_rate="auto", random_state=...)` y `.fit_transform(X)`: `mapa_a` con semilla 0, `mapa_b` con la semilla 0 **otra vez** y `mapa_c` con semilla 1.
2. Pregunta con `np.allclose(...)` si `mapa_a` es igual a `mapa_b` (guárdalo en `igual_ab`) y si `mapa_a` es igual a `mapa_c` (en `igual_ac`).
3. Mide qué tan bien conserva cada mapa a los vecinos con `trustworthiness(X, mapa, n_neighbors=10)` (1 = perfecto). Guarda `confianza_a` y `confianza_c`.
4. Comprueba: misma semilla da el mismo mapa, otra semilla da otro mapa, pero la confianza de los dos casi no cambia (difieren menos de 0.03).

In [ ]:
# Escribe tu código aquí

from sklearn.manifold import trustworthiness

# 1) Tres mapas: semilla 0, semilla 0 otra vez y semilla 1
# mapa_a = TSNE(n_components=2, perplexity=30, init="random", learning_rate="auto", random_state=0).fit_transform(X)
# mapa_b = ...
# mapa_c = ...

# 2) ¿Son iguales?
# igual_ab = np.allclose(mapa_a, mapa_b)
# igual_ac = np.allclose(mapa_a, ...)

# 3) ¿Cuantos vecinos conserva cada mapa?
# confianza_a = trustworthiness(X, mapa_a, n_neighbors=10)
# confianza_c = ...

# 4) Comprobaciones
# assert igual_ab and not igual_ac
# assert abs(confianza_a - confianza_c) < 0.03
# print(...)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
from sklearn.manifold import trustworthiness

# 1) Tres mapas: semilla 0, semilla 0 otra vez y semilla 1
def hacer_mapa(semilla):
    return TSNE(n_components=2, perplexity=30, init="random", learning_rate="auto", random_state=semilla).fit_transform(X)

mapa_a, mapa_b, mapa_c = hacer_mapa(0), hacer_mapa(0), hacer_mapa(1)

# 2) ¿Son iguales?
igual_ab = np.allclose(mapa_a, mapa_b)
igual_ac = np.allclose(mapa_a, mapa_c)

# 3) ¿Cuantos vecinos conserva cada mapa? (1 = todos)
confianza_a = trustworthiness(X, mapa_a, n_neighbors=10)
confianza_c = trustworthiness(X, mapa_c, n_neighbors=10)

# 4) Comprobaciones
assert igual_ab and not igual_ac
assert abs(confianza_a - confianza_c) < 0.03
print(f"Misma semilla, mismo mapa: {igual_ab} | otra semilla, mismo mapa: {igual_ac}")
print(f"Confianza con la semilla 0: {confianza_a:.3f} | con la semilla 1: {confianza_c:.3f}")
print("Los amigos de cada quien casi no cambian; lo que cambia es en que rincon del salon queda cada mesa.")
```
</details>

---


---
## Resumen en una frase 🎯

> ***t-SNE* dibuja en un plano datos con muchas columnas intentando que cada punto quede junto a sus «amigos»; sirve para explorar y encontrar grupos, pero el tamaño de los grupos y la distancia entre ellos no significan nada, el mapa cambia con la semilla y la perplejidad, y no se puede reutilizar con datos nuevos.**

### Lo que aprendiste hoy:
- ✅ *t-SNE* conserva **vecindades** (quién está cerca de quién), no las distancias grandes.
- ✅ La **perplejidad** es «cuántos amigos cuenta cada punto»: **prueba varias** y confía en lo que se repite.
- ✅ **Tamaños** y **distancias entre grupos** del mapa **no se interpretan**.
- ✅ Cambia con la **semilla** y puede inventar «grupos» en puro azar.
- ✅ **No hay transformación** para datos nuevos: sirve para **mirar**, no para entrenar modelos.

> 🎓 **Siguiente paso:** [Isomap y variedades (Dummies)](02_Isomap_Variedades_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../01_tSNE.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué significa la perplejidad de *t-SNE*? ¿Qué le pasa al mapa si es muy baja y si es muy alta?
2. En un mapa de *t-SNE* de tus clientes ves tres grupos: el grupo A es el doble de grande que el B y está muy lejos del C. Tu jefe concluye que «A es más variado y muy distinto de C». ¿Qué le responderías y qué harías antes de creerlo?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>